**ReadME**  
**This Version Does not contain any test Cases, Please refer to V1 or V2 for detailed Test Case Output**  
* V1: 25 Historical Quarters Data (5 Assets)
* V2: 100 Historial Quarters Data (5 Assets)

**Overview of the Entire JupyterNotebook**  
Section 1: Required Libraries  
Section 2: Data Processing
* 2.1 Single Asset Data Processing  
* 2.2 Single Asset Test Case  
* 2.3 Multi Asset Data Processing  
* 2.4 Multi Asset Test Case

Section 3: LSTM Machine Learning Estimator  
* 3.1 Pre-Processing
* 3.2 Pre-Processing Test Case
* 3.3 LSTM Model SetUp
* 3.4 LSTM Test Case

Section 4: ChatGPT 15 Asset Main Case

**Use Intruction**  
Please run all the sections with functions to construct the framework, Test Case Sections are optional to run

**Section 1: Required Libraries**

In [ ]:
import pandas as pd
import yfinance as yf
import numpy as np

from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Dense
from tensorflow.keras.regularizers import l2

**Section 2: Data Processing**  
Section 2.1 Single Asset Data Processing

In [ ]:
def get_historical_returns(ticker, start_date, end_date, frequency="monthly"):
    'Function to fetch Historical Price data and compute returns'

    data = yf.download(ticker,start=start_date, end=end_date)

    # Calculate Daily Returns
    daily_data = data.copy()
    daily_data['Return'] = daily_data['Close'].pct_change()
    daily_returns = daily_data[['Return']].dropna()

    # Calculate Monthly Returns
    monthly_data = data.copy()
    monthly_data['Return'] = monthly_data['Close']
    monthly_data = monthly_data['Return'].resample('M').last()
    monthly_returns = monthly_data.pct_change()
    monthly_returns = monthly_returns.dropna()

    if frequency == "daily": return daily_returns
    if frequency == "monthly": return monthly_returns

    return monthly_data

def resample_quaterly_data(quaterly_data, target_data):
    'Repeat the quaterly available ratios to same frequency as target return'

    quaterly_data.index = pd.to_datetime(quaterly_data.index)
    target_data.index = pd.to_datetime(target_data.index)

    # Resample the quaterly data to daily frequency using Forward Fill
    quaterly_data.index = quaterly_data.index + pd.DateOffset(days=1)
    aligned_quaterly_data = quaterly_data.reindex(target_data.index, method='ffill')

    aligned_quaterly_data = aligned_quaterly_data.dropna()
    return aligned_quaterly_data


def load_features(path_to_file, ticker, start_date, end_date):
    'Function to Load all features for a single company'

    # Load the Excel file and read Data from the file
    file_path = path_to_file + ticker + '.xlsx'
    sheet_name = ticker + '-US'
    data = pd.read_excel(file_path, sheet_name=sheet_name, engine='openpyxl')

    # Remove rows with any NaN values
    # Because time frame is longer, cannot apply this
    # data = data.dropna()

    # Reset the index of the DataFrame and drop the old index
    data = data.reset_index(drop=True)

    data = data.set_index('Date').T
    data.index = pd.to_datetime(data.index, format='%b \'%y')
    data.index = data.index + pd.offsets.MonthEnd()
    ratio_data = data.apply(pd.to_numeric)

    # Select a few columns
    pe_column = 'Price/Earnings'
    pb_column = 'Price/Book Value'
    roa_column = 'Return on Assets'
    roe_column = 'Return on Equity '
    fcf_column = 'Free Cash Flow per Share'
    ratio_data = data[[pe_column, pb_column, roa_column, roe_column, fcf_column]]

    # Drop N/A dates
    # Removing rows with any NaN values
    ratio_data = ratio_data.dropna()

    # Process Return Data
    returns_data = get_historical_returns(ticker, start_date, end_date)
    adjusted_ratio_data = resample_quaterly_data(ratio_data, returns_data)
    features = pd.concat([adjusted_ratio_data, returns_data],axis=1)

    return features

Section 2.3 Multi Asset Data Processing

In [ ]:
def multi_df(path_to_file, ticker_list, start_date, end_date):
    company_data = {}
    for ticker in ticker_list:
        company_data[ticker] = load_features(path_to_file, ticker, start_date, end_date)

    # Initialize a list to hold DataFrames with the new multi-index
    multi_index_dfs = []

    for company, df in company_data.items():
        # Set the company name as an additional level in the index
        df_multi_index = df.copy()
        df_multi_index['Company'] = company
        df_multi_index.set_index(['Company', df_multi_index.index], inplace=True)

        # Append to the list
        multi_index_dfs.append(df_multi_index)

    # Concatenate all DataFrames into a single multi-index DataFrame
    final_df = pd.concat(multi_index_dfs)

    return final_df

**Section 3: LSTM Machine Learning Estimator**  
Section 3.1 Pre-Processing

In [ ]:
def create_sequences(features, targets, seq_length):
    'Function to create sequence'
    'Need to define the sequence length: e.g. using 4 quaters to predict the next quater'

    xs = []
    ys = []

    for i in range(len(features)-seq_length):
        x = features[i:(i+seq_length)]
        y = targets.iloc[i+seq_length]
        xs.append(x)
        ys.append(y)

    return np.array(xs), np.array(ys)

Section 3.3 LSTM Model SetUp

In [ ]:
# LSTM Model Set Up

# Model architecture
model = tf.keras.Sequential([
    LSTM(512, return_sequences=True),
    Dropout(0.02),
    LSTM(256, return_sequences=True),
    LSTM(128),
    Dense(1, activation='linear', kernel_regularizer=l2(0.0005))
])

# Compile the model
optimizer = tf.keras.optimizers.Adam(learning_rate=0.005)
model.compile(optimizer=optimizer, loss='mean_squared_error')

**Section 4: ChatGPT 15 Stock Test Case**

Step 1: Define Input and Parameters

In [ ]:
# 1. File Path
path_to_file = ''
# 2. Ticker List
ChatGPT_stocks = ['AAPL', 'MSFT', 'AMZN', 'GOOGL', 'JNJ', 'V', 'PG', 'JPM', 'UNH', 'MA', 'INTC', 'VZ', 'GOOG', 'HD', 'T']
# 3. Target Time Frame
start_date = '2016-09-30'
end_date = '2021-09-30'
# 4. Sequence Length
seq_length = 6
# 5. Training and Validation Set Split Ratio
train_ratio = 0.8
# 6. Num Epoch and Num Batch
num_epoch = 20
num_batch = 12

Step2: Pre-Processing

In [ ]:
# Loading Phase: Took a while to run this (Don't Rerun)
final_df = multi_df(path_to_file, ChatGPT_stocks, start_date, end_date)

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%*******

In [ ]:
multi_df_gpt = final_df
multi_return_gpt = pd.DataFrame(multi_df_gpt['Return'])
# print(multi_df_gpt)
# print(multi_return_gpt)

In [ ]:
features = multi_df_gpt
targets = multi_return_gpt
multi_X_gpt, multi_y_gpt = create_sequences(features, targets, seq_length)
# print(multi_X_test1)
# print(multi_y_test1)

Step3: Train Model on Training and Validation Sets

In [ ]:
# Variable in Use and Constant Define
X = multi_X_gpt
y = multi_y_gpt

# Split data into training and validation sets
train_size = int(len(X) * train_ratio)
X_train, X_vali = X[:train_size], X[train_size:]
y_train, y_vali = y[:train_size], y[train_size:]

# Train the model
model.fit(X_train, y_train, epochs=num_epoch, batch_size=num_batch, validation_data=(X_vali, y_vali))

Epoch 1/20
60/60 [==============================] - 1s 17ms/step - loss: 0.0047 - val_loss: 0.0042
Epoch 2/20
60/60 [==============================] - 1s 16ms/step - loss: 0.0049 - val_loss: 0.0042
Epoch 3/20
60/60 [==============================] - 1s 16ms/step - loss: 0.0048 - val_loss: 0.0046
Epoch 4/20
60/60 [==============================] - 1s 16ms/step - loss: 0.0048 - val_loss: 0.0043
Epoch 5/20
60/60 [==============================] - 1s 16ms/step - loss: 0.0047 - val_loss: 0.0043
Epoch 6/20
60/60 [==============================] - 1s 16ms/step - loss: 0.0047 - val_loss: 0.0044
Epoch 7/20
60/60 [==============================] - 1s 16ms/step - loss: 0.0046 - val_loss: 0.0042
Epoch 8/20
60/60 [==============================] - 1s 16ms/step - loss: 0.0046 - val_loss: 0.0042
Epoch 9/20
60/60 [==============================] - 1s 16ms/step - loss: 0.0046 - val_loss: 0.0042
Epoch 10/20
60/60 [==============================] - 1s 16ms/step - loss: 0.0046 - val_loss: 0.0042
Epoch 11/

In [ ]:
# Make predictions
X_pred, y_pred = X_train, y_train

predictions = model.predict(X_pred)
print(predictions)

23/23 [==============================] - 1s 6ms/step
[[0.01727694]
 [0.01737933]
 [0.01741524]
 [0.01746609]
 [0.01751982]
 [0.01752711]
 [0.01757676]
 [0.01766689]
 [0.01774192]
 [0.01779491]
 [0.01799333]
 [0.01810281]
 [0.01821443]
 [0.01881543]
 [0.01913609]
 [0.01936607]
 [0.02026843]
 [0.02067448]
 [0.02092688]
 [0.01947551]
 [0.01876552]
 [0.01835045]
 [0.01892677]
 [0.01928201]
 [0.01950077]
 [0.0201197 ]
 [0.02048667]
 [0.02059587]
 [0.02105879]
 [0.02128842]
 [0.02137804]
 [0.02219971]
 [0.02251833]
 [0.02265929]
 [0.02264612]
 [0.02269579]
 [0.02270824]
 [0.02339823]
 [0.0235867 ]
 [0.02366991]
 [0.02394355]
 [0.02401373]
 [0.02404449]
 [0.0241152 ]
 [0.02413747]
 [0.02414138]
 [0.02400995]
 [0.02400409]
 [0.02400117]
 [0.02393467]
 [0.02389851]
 [0.02384657]
 [0.02371623]
 [0.02360825]
 [0.02354008]
 [0.02140841]
 [0.02015601]
 [0.01939396]
 [0.01900553]
 [0.0187667 ]
 [0.01856081]
 [0.01820991]
 [0.01807568]
 [0.01804691]
 [0.0182007 ]
 [0.01830129]
 [0.01837426]
 [0.01998

In [ ]:
print(X_train)

[[[ 1.38706590e+01  4.59765200e+00  1.44827640e+01  3.46946370e+01
    2.48503400e+00  4.33434631e-03]
  [ 1.38706590e+01  4.59765200e+00  1.44827640e+01  3.46946370e+01
    2.48503400e+00 -2.65985930e-02]
  [ 1.38706590e+01  4.59765200e+00  1.44827640e+01  3.46946370e+01
    2.48503400e+00  4.79551503e-02]
  [ 1.68023390e+01  5.57768600e+00  1.42948910e+01  3.45733520e+01
    2.52979700e+00  4.77464928e-02]
  [ 1.68023390e+01  5.57768600e+00  1.42948910e+01  3.45733520e+01
    2.52979700e+00  1.28883455e-01]
  [ 1.68023390e+01  5.57768600e+00  1.42948910e+01  3.45733520e+01
    2.52979700e+00  4.86896701e-02]]

 [[ 1.38706590e+01  4.59765200e+00  1.44827640e+01  3.46946370e+01
    2.48503400e+00 -2.65985930e-02]
  [ 1.38706590e+01  4.59765200e+00  1.44827640e+01  3.46946370e+01
    2.48503400e+00  4.79551503e-02]
  [ 1.68023390e+01  5.57768600e+00  1.42948910e+01  3.45733520e+01
    2.52979700e+00  4.77464928e-02]
  [ 1.68023390e+01  5.57768600e+00  1.42948910e+01  3.45733520e+01
    

Step4: Evaluate the Model on Test Set

In [ ]:
# Test Period 1: September 30 2021 to July 30 2023
# Test Period 2: March 14, 2023 to July 31 2023
# Test Period 3: May 01 2023 to July 31 2023

start_t1 = '2021-09-30'
end_t1 = '2023-07-31'

start_t2 = '2023-03-14'
end_t2 = '2023-07-31'

start_t3 = '2023-05-01'
end_t3 = '2023-07-31'

In [ ]:
# Loading Test Phase: Took a while to run this (Don't Rerun)
final_df_t1 = multi_df(path_to_file, ChatGPT_stocks, start_t1, end_t1)
final_df_t2 = multi_df(path_to_file, ChatGPT_stocks, start_t2, end_t2)
final_df_t3 = multi_df(path_to_file, ChatGPT_stocks, start_t3, end_t3)

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%*******

In [ ]:
multi_df_t1 = final_df_t1
multi_return_t1 = pd.DataFrame(multi_df_t1['Return'])
seq_length = 6
multi_X_t1, multi_y_t1 = create_sequences(multi_df_t1, multi_return_t1, seq_length)

multi_df_t2 = final_df_t2
multi_return_t2 = pd.DataFrame(multi_df_t2['Return'])
seq_length = 6
multi_X_t2, multi_y_t2 = create_sequences(multi_df_t2, multi_return_t2, seq_length)

multi_df_t3 = final_df_t3
multi_return_t3 = pd.DataFrame(multi_df_t3['Return'])
seq_length = 6
multi_X_t3, multi_y_t3 = create_sequences(multi_df_t3, multi_return_t3, seq_length)

In [ ]:
# Evaluate the model
X_test, y_test = multi_X_t1, multi_y_t1
loss = model.evaluate(X_test, y_test)
print(f"Mean Squared Error: {loss}")

11/11 [==============================] - 0s 6ms/step - loss: nan
Mean Squared Error: nan


In [ ]:
# Make predictions
X_pred, y_pred = multi_X_t1, multi_y_t1

predictions = model.predict(X_pred)
print(predictions)

11/11 [==============================] - 0s 9ms/step
[[0.02373397]
 [0.02352534]
 [0.02342271]
 [0.02337022]
 [0.02334863]
 [0.02332986]
 [0.0233232 ]
 [0.0233184 ]
 [0.02332009]
 [0.02332627]
 [0.02345863]
 [0.02348835]
 [0.02351201]
 [0.02365153]
 [0.02378841]
 [0.02382642]
 [0.02368634]
 [0.02379728]
 [0.02364001]
 [0.0234861 ]
 [0.02324045]
 [0.02304095]
 [0.02268235]
 [0.02242917]
 [0.02238704]
 [0.02240903]
 [0.02209843]
 [0.02199151]
 [0.02194091]
 [0.02175859]
 [0.02161062]
 [0.02154405]
 [0.02167977]
 [0.02163775]
 [0.02154238]
 [0.02166743]
 [0.0216505 ]
 [0.02160604]
 [0.02141756]
 [0.02149994]
 [0.02133481]
 [0.02128107]
 [0.02191915]
 [0.02207933]
 [0.02228464]
 [0.02311392]
 [0.0237521 ]
 [0.02456883]
 [0.02570163]
 [0.02760616]
 [0.02939897]
 [0.0302218 ]
 [0.03051632]
 [0.03095947]
 [0.03126603]
 [0.03143048]
 [0.03167873]
 [0.03214612]
 [0.03198713]
 [0.03140356]
 [0.02716476]
 [0.022367  ]
 [0.02087322]
 [0.02049716]
 [0.02033622]
 [0.02043434]
 [0.02060157]
 [0.02002

In [ ]:
print(y_pred)

[[-0.09713079]
 [-0.05588327]
 [-0.08142969]
 [ 0.18863365]
 [-0.0325518 ]
 [-0.120977  ]
 [ 0.10955137]
 [-0.03462891]
 [-0.12227255]
 [ 0.11052106]
 [ 0.02162319]
 [ 0.1186486 ]
 [ 0.02898726]
 [ 0.04461344]
 [ 0.09433005]
 [ 0.00958911]
 [ 0.17629107]
 [-0.00310596]
 [ 0.01733268]
 [-0.0753449 ]
 [-0.03919867]
 [ 0.03186181]
 [-0.09986705]
 [-0.02035887]
 [-0.05532059]
 [ 0.09309662]
 [-0.06863999]
 [-0.10926686]
 [-0.00330609]
 [ 0.09912546]
 [-0.06004543]
 [ 0.03331661]
 [ 0.00649692]
 [ 0.1558816 ]
 [ 0.06576491]
 [ 0.06876913]
 [ 0.03699867]
 [-0.00637227]
 [ 0.02660246]
 [ 0.0399237 ]
 [-0.04925197]
 [-0.10282991]
 [ 0.02667252]
 [ 0.06143729]
 [-0.23752509]
 [-0.03276432]
 [-0.11645921]
 [ 0.27059597]
 [-0.06061505]
 [-0.10862189]
 [-0.09345131]
 [-0.0575947 ]
 [-0.12989435]
 [ 0.22773806]
 [-0.08629879]
 [ 0.09614769]
 [ 0.02091196]
 [ 0.14348037]
 [ 0.08110797]
 [ 0.01419152]
 [ 0.10749881]
 [-0.041531  ]
 [ 0.02082135]
 [-0.06591906]
 [-0.00182176]
 [ 0.02969485]
 [-0.17946

In [ ]:
# Evaluate the model
X_test, y_test = multi_X_t2, multi_y_t2
loss = model.evaluate(X_test, y_test)
print(f"Mean Squared Error: {loss}")

2/2 [==============================] - 0s 6ms/step - loss: nan
Mean Squared Error: nan


In [ ]:
# Make predictions
X_pred, y_pred = multi_X_t1, multi_y_t1

predictions = model.predict(X_pred)
print(predictions)

11/11 [==============================] - 0s 7ms/step
[[0.02373397]
 [0.02352534]
 [0.02342271]
 [0.02337022]
 [0.02334863]
 [0.02332986]
 [0.0233232 ]
 [0.0233184 ]
 [0.02332009]
 [0.02332627]
 [0.02345863]
 [0.02348835]
 [0.02351201]
 [0.02365153]
 [0.02378841]
 [0.02382642]
 [0.02368634]
 [0.02379728]
 [0.02364001]
 [0.0234861 ]
 [0.02324045]
 [0.02304095]
 [0.02268235]
 [0.02242917]
 [0.02238704]
 [0.02240903]
 [0.02209843]
 [0.02199151]
 [0.02194091]
 [0.02175859]
 [0.02161062]
 [0.02154405]
 [0.02167977]
 [0.02163775]
 [0.02154238]
 [0.02166743]
 [0.0216505 ]
 [0.02160604]
 [0.02141756]
 [0.02149994]
 [0.02133481]
 [0.02128107]
 [0.02191915]
 [0.02207933]
 [0.02228464]
 [0.02311392]
 [0.0237521 ]
 [0.02456883]
 [0.02570163]
 [0.02760616]
 [0.02939897]
 [0.0302218 ]
 [0.03051632]
 [0.03095947]
 [0.03126603]
 [0.03143048]
 [0.03167873]
 [0.03214612]
 [0.03198713]
 [0.03140356]
 [0.02716476]
 [0.022367  ]
 [0.02087322]
 [0.02049716]
 [0.02033622]
 [0.02043434]
 [0.02060157]
 [0.02002

In [ ]:
print(y_pred)

[[-0.09713079]
 [-0.05588327]
 [-0.08142969]
 [ 0.18863365]
 [-0.0325518 ]
 [-0.120977  ]
 [ 0.10955137]
 [-0.03462891]
 [-0.12227255]
 [ 0.11052106]
 [ 0.02162319]
 [ 0.1186486 ]
 [ 0.02898726]
 [ 0.04461344]
 [ 0.09433005]
 [ 0.00958911]
 [ 0.17629107]
 [-0.00310596]
 [ 0.01733268]
 [-0.0753449 ]
 [-0.03919867]
 [ 0.03186181]
 [-0.09986705]
 [-0.02035887]
 [-0.05532059]
 [ 0.09309662]
 [-0.06863999]
 [-0.10926686]
 [-0.00330609]
 [ 0.09912546]
 [-0.06004543]
 [ 0.03331661]
 [ 0.00649692]
 [ 0.1558816 ]
 [ 0.06576491]
 [ 0.06876913]
 [ 0.03699867]
 [-0.00637227]
 [ 0.02660246]
 [ 0.0399237 ]
 [-0.04925197]
 [-0.10282991]
 [ 0.02667252]
 [ 0.06143729]
 [-0.23752509]
 [-0.03276432]
 [-0.11645921]
 [ 0.27059597]
 [-0.06061505]
 [-0.10862189]
 [-0.09345131]
 [-0.0575947 ]
 [-0.12989435]
 [ 0.22773806]
 [-0.08629879]
 [ 0.09614769]
 [ 0.02091196]
 [ 0.14348037]
 [ 0.08110797]
 [ 0.01419152]
 [ 0.10749881]
 [-0.041531  ]
 [ 0.02082135]
 [-0.06591906]
 [-0.00182176]
 [ 0.02969485]
 [-0.17946